# 04 Generate Uniform Subset DII Intermediates

Purpose: generate the uniform-subset resampling weights, point/cache files, and DII/weight provenance files for the SOCAT-vs-uniform subset three-panel figure.

Inputs:
- pCO2 LEAP reconstruction zarr
- SOCAT mask zarr
- RECCAP2 region mask NetCDF

Outputs:
- `SOCAT2_spatial_with_OTweights.csv`
- `Uniform_split_spatial.csv`
- `SOCAT1_points.csv`
- `SOCAT2_resampled_points.csv`
- `Finalweights_SOCAT1.txt`
- `Finalimbs_SOCAT1.txt`
- `standardimbs_SOCAT1_my_pipeline.txt`
- `imbs_weightedSOCAT2_fromSOCAT1.txt`

Estimated runtime: slow. This notebook first generates the resampling weights/cache, then learns feature weights on SOCAT set 1 and evaluates DII on SOCAT set 1 and weighted SOCAT set 2.

Notes:
- This notebook is provenance material for the SOCAT-vs-uniform subset figures.
- The resampling weights and resampled point cache are generated here so the DII provenance does not depend on a plotting script.
- This notebook generates the resampling and DII outputs in one self-contained workflow.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from dadapy.feature_weighting import FeatureWeighting
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

RECONSTRUCTION_ZARR = Path("../raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr")
SOCAT_MASK_ZARR = Path("../raw/socat_mask_feb1982-dec2022.zarr")
REGION_MASK_FILE = Path("../raw/RECCAP2_region_masks_all_v20221025.nc")
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_SLICE = slice("2020-01-01", "2022-12-31")
SOCAT1_FRACTION = 0.5
SOCAT1_RANDOM_STATE = 67
UNIFORM_RANDOM_STATE = 10
RESAMPLE_RANDOM_STATE = 42
N_EPOCHS = 80


## Feature Definitions


In [ ]:
FEATURES = [
    "sst",
    "sst_anomaly",
    "sss",
    "sss_anomaly",
    "chl_log",
    "chl_log_anomaly",
    "mld_log",
    "xco2_trend",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"


## Helper Functions


In [ ]:
def ranks_from_distances(distances):
    distances = np.asarray(distances)
    n_points = distances.shape[0]
    ranks = np.full((n_points, n_points), -1, dtype=np.int32)

    for i in range(n_points):
        mask = np.ones(n_points, dtype=bool)
        mask[i] = False
        neighbors = np.arange(n_points)[mask]
        order = neighbors[np.argsort(distances[i, mask])]
        ranks[i, order] = np.arange(1, n_points, dtype=np.int32)

    return ranks

def weighted_info_imbalance_first_nn(ranks_a, ranks_b, weights, first_rank=1):
    ranks_a = np.asarray(ranks_a)
    ranks_b = np.asarray(ranks_b)
    weights = np.asarray(weights, dtype=float)

    n_points = ranks_a.shape[0]
    cumulative_b = np.zeros((n_points, n_points), dtype=float)

    for i in range(n_points):
        neighbor_index = np.where(ranks_b[i] > 0)[0]
        ordered = neighbor_index[np.argsort(ranks_b[i, neighbor_index])]
        cumulative_b[i, ordered] = np.cumsum(weights[ordered])

    first_a = ranks_a == first_rank
    numerator = np.sum((weights[:, None] * cumulative_b)[first_a])
    return 2.0 * numerator / np.sum(weights)

def add_lat_lon(frame):
    out = frame.copy()
    x = out["A"].to_numpy()
    y = out["B"].to_numpy()
    z = out["C"].to_numpy()
    radius = np.sqrt(x * x + y * y + z * z)
    x, y, z = x / radius, y / radius, z / radius
    out["lat_deg"] = np.degrees(np.arcsin(x))
    out["lon_deg"] = (np.degrees(-np.arctan2(y, z))) % 360 - 180
    return out

def pairwise_sqeuclidean(x, y):
    return (x**2).sum(axis=1)[:, None] + (y**2).sum(axis=1)[None, :] - 2 * x @ y.T

def unbalanced_sinkhorn(a, b, cost, epsilon=0.15, tau_a=0.5, tau_b=0.5, max_iter=250):
    kernel = np.exp(-cost / epsilon)
    kernel = np.maximum(kernel, 1e-300)
    u = np.ones_like(a)
    v = np.ones_like(b)
    alpha = tau_a / (tau_a + epsilon)
    beta = tau_b / (tau_b + epsilon)
    for _ in range(max_iter):
        u = (a / (kernel @ v + 1e-300)) ** alpha
        v = (b / (kernel.T @ u + 1e-300)) ** beta
    return (u[:, None] * kernel) * v[None, :]

def add_uniform_weights(source, target):
    x_source = source[["A", "B", "C"]].to_numpy()
    x_target = target[["A", "B", "C"]].to_numpy()
    rng = np.random.default_rng(0)
    ns = min(len(x_source), 2000)
    nt = min(len(x_target), 2000)
    idx_source = rng.choice(len(x_source), size=ns, replace=False)
    idx_target = rng.choice(len(x_target), size=nt, replace=False)

    cost = pairwise_sqeuclidean(x_source[idx_source], x_target[idx_target])
    a = np.ones(ns) / ns
    b = np.ones(nt) / nt
    transport = unbalanced_sinkhorn(a, b, cost)
    weights_sub = transport.sum(axis=1) / a

    knn = KNeighborsRegressor(n_neighbors=15, weights="distance")
    knn.fit(x_source[idx_source], weights_sub)
    weights_uot = knn.predict(x_source)
    weights_uot = np.clip(weights_uot, 0, np.quantile(weights_uot, 0.99))
    weights_uot = weights_uot / weights_uot.mean()

    q = np.linspace(0, 1, 17)
    edges = [np.quantile(x_target[:, j], q) for j in range(x_target.shape[1])]

    def digitize_cols(values):
        indexes = []
        for j, edge in enumerate(edges):
            bins = np.digitize(values[:, j], edge[1:-1], right=False)
            indexes.append(np.clip(bins, 0, len(edge) - 2))
        return indexes

    source_bins = digitize_cols(x_source)
    target_bins = digitize_cols(x_target)

    def hist1d(indexes):
        hist = np.bincount(indexes, minlength=16).astype(float)
        return hist / hist.sum()

    target_hists = [hist1d(indexes) for indexes in target_bins]
    weights_ipf = weights_uot.copy()
    for _ in range(8):
        for source_index, target_hist in zip(source_bins, target_hists):
            source_hist = np.bincount(source_index, weights=weights_ipf, minlength=16)
            source_hist = source_hist / source_hist.sum()
            scale = (target_hist + 1e-6) / (source_hist + 1e-6)
            weights_ipf *= np.clip(scale, 0.25, 4.0)[source_index]
        weights_ipf = np.clip(weights_ipf, 0, np.quantile(weights_ipf, 0.995))
        weights_ipf = weights_ipf / weights_ipf.mean()

    out = source.copy()
    out["uot_weight"] = weights_uot
    out["final_weight"] = weights_ipf
    return out

def load_socat_scaled(reconstruction_zarr, socat_mask_zarr):
    socat_mask = xr.open_dataset(socat_mask_zarr, engine="zarr")
    ds = xr.open_dataset(reconstruction_zarr, engine="zarr")
    ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))

    aligned_mask = socat_mask.reindex(time=ds.time, method="nearest")
    ds = ds.where(aligned_mask.socat_mask == 1)
    ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
    ds = ds[FEATURES + [TARGET]].sel(time=TIME_SLICE)
    socat = ds.to_dataframe().dropna()

    scaler = StandardScaler()
    scaled = scaler.fit_transform(socat.loc[:, FEATURES + [TARGET]])
    return pd.DataFrame(scaled, columns=FEATURES + [TARGET]), scaler

def rank_matrix(values):
    feature_weighting = FeatureWeighting(values, verbose=True)
    return ranks_from_distances(feature_weighting.full_distance_matrix)


## Load SOCAT and Recreate the SOCAT1/SOCAT2 Split

This split matches `Comparison_SOCAT_uniform.ipynb`.


In [ ]:
SOCAT_scaled, SOCAT_scaler = load_socat_scaled(RECONSTRUCTION_ZARR, SOCAT_MASK_ZARR)
SOCAT1_scaled = SOCAT_scaled.sample(frac=SOCAT1_FRACTION, random_state=SOCAT1_RANDOM_STATE)
SOCAT2_scaled = SOCAT_scaled.drop(SOCAT1_scaled.index)

SOCAT_scaled.shape, SOCAT1_scaled.shape, SOCAT2_scaled.shape


## Generate Uniform-Reweighting Cache and Resampled Points

This reproduces the resampling-weight part of `Comparison_SOCAT_uniform.ipynb`. It saves the weights used for DII and the point caches used by the map panels.


In [ ]:
ds = xr.open_dataset(RECONSTRUCTION_ZARR, engine="zarr")
ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))
ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
ds = ds[FEATURES + [TARGET]]

oceanmask = xr.open_dataset(REGION_MASK_FILE).open_ocean
oceanmask = oceanmask.roll(lon=180, roll_coords="lon")
oceanmask["lon"] = np.arange(-179.5, 180.5, 1)
oceanmask = oceanmask.rename({"lon": "xlon", "lat": "ylat"})

dfocean = ds.where(oceanmask != 0).sel(time=TIME_SLICE)
uniform = dfocean.to_dataframe().sample(n=22000, random_state=UNIFORM_RANDOM_STATE)
uniform_scaled = pd.DataFrame(
    SOCAT_scaler.transform(uniform.loc[:, FEATURES + [TARGET]]),
    columns=FEATURES + [TARGET],
)[["A", "B", "C"]].dropna()

weighted = add_uniform_weights(SOCAT2_scaled[["A", "B", "C"]], uniform_scaled)
weighted.to_csv(OUTPUT_DIR / "SOCAT2_spatial_with_OTweights.csv", index=False)
uniform_scaled.to_csv(OUTPUT_DIR / "Uniform_split_spatial.csv", index=False)

weights = weighted["final_weight"].to_numpy()
probabilities = weights / weights.sum()
rng = np.random.default_rng(RESAMPLE_RANDOM_STATE)
idx = rng.choice(len(weighted), size=len(weighted), replace=True, p=probabilities)

spatial_columns = ["A", "B", "C"]
spatial_indexes = [FEATURES.index(col) for col in spatial_columns]

# The resampling indices are positions within SOCAT2.
socat2_resampled_inv = SOCAT_scaler.inverse_transform(SOCAT2_scaled.iloc[idx, :])
socat2_resampled_points = pd.DataFrame(socat2_resampled_inv[:, spatial_indexes], columns=spatial_columns)
add_lat_lon(socat2_resampled_points).to_csv(OUTPUT_DIR / "SOCAT2_resampled_points.csv", index=False)

socat1_inv = SOCAT_scaler.inverse_transform(SOCAT_scaled.iloc[SOCAT1_scaled.index, :])
socat1_points = pd.DataFrame(socat1_inv[:, spatial_indexes], columns=spatial_columns)
add_lat_lon(socat1_points).to_csv(OUTPUT_DIR / "SOCAT1_points.csv", index=False)

sampling_weights = weighted["final_weight"].to_numpy()


## Learn Feature Weights on SOCAT1

Slow cell. Generates `Finalweights_SOCAT1.txt` and `Finalimbs_SOCAT1.txt`.


In [ ]:
target = FeatureWeighting(SOCAT1_scaled[[TARGET]].to_numpy(), verbose=True)
inputs = FeatureWeighting(SOCAT1_scaled[FEATURES].to_numpy(), verbose=True)

final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
    target_data=target,
    initial_weights=None,
    n_epochs=N_EPOCHS,
    learning_rate=None,
    decaying_lr="cos",
)

np.savetxt(OUTPUT_DIR / "Finalweights_SOCAT1.txt", final_weights)
np.savetxt(OUTPUT_DIR / "Finalimbs_SOCAT1.txt", final_imbs)


## Compute Weighted DII Curve on SOCAT2

This evaluates the SOCAT1-learned weights on SOCAT2 using the uniform-reweighting probabilities.


In [ ]:
ranks_target_socat2 = rank_matrix(SOCAT2_scaled[[TARGET]].to_numpy())
normalized_sampling_weights = sampling_weights / sampling_weights.sum()

imbs_weighted_socat2 = []
for k, weights in enumerate(final_weights):
    if np.any(np.isnan(weights)):
        print("NaNs encountered for index", k)
        imbs_weighted_socat2.append(np.nan)
        continue

    ranks_source = rank_matrix(SOCAT2_scaled[FEATURES].to_numpy() * weights)
    imbalance = weighted_info_imbalance_first_nn(
        ranks_source,
        ranks_target_socat2,
        normalized_sampling_weights,
        first_rank=1,
    )
    print(imbalance)
    imbs_weighted_socat2.append(imbalance)

np.savetxt(OUTPUT_DIR / "imbs_weightedSOCAT2_fromSOCAT1.txt", imbs_weighted_socat2)


## Compute Standard DII Curve on SOCAT1

This evaluates the same SOCAT1-learned weights on SOCAT1 with uniform sample weights.


In [ ]:
ranks_target_socat1 = rank_matrix(SOCAT1_scaled[[TARGET]].to_numpy())
uniform_weights = np.ones(len(SOCAT1_scaled))
uniform_weights = uniform_weights / uniform_weights.sum()

imbs_socat1 = []
for k, weights in enumerate(final_weights):
    if np.any(np.isnan(weights)):
        print("NaNs encountered for index", k)
        imbs_socat1.append(np.nan)
        continue

    ranks_source = rank_matrix(SOCAT1_scaled[FEATURES].to_numpy() * weights)
    imbalance = weighted_info_imbalance_first_nn(
        ranks_source,
        ranks_target_socat1,
        uniform_weights,
        first_rank=1,
    )
    print(imbalance)
    imbs_socat1.append(imbalance)

np.savetxt(OUTPUT_DIR / "standardimbs_SOCAT1_my_pipeline.txt", imbs_socat1)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = [
    "SOCAT2_spatial_with_OTweights.csv",
    "Uniform_split_spatial.csv",
    "SOCAT1_points.csv",
    "SOCAT2_resampled_points.csv",
    "Finalweights_SOCAT1.txt",
    "Finalimbs_SOCAT1.txt",
    "standardimbs_SOCAT1_my_pipeline.txt",
    "imbs_weightedSOCAT2_fromSOCAT1.txt",
]

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
